# G1 收尾审查：只读取 Google Drive 已有结果

**旧 runtime 断开没有影响。可使用全新的 CPU runtime，无需 GPU、旧 `/content` 文件或旧 Python 变量。**

输入固定为 Drive 中 `CEG-WM/gds-v21/runs/g1-v21-20261128-8sources-48attempts/`：报告、CSV、48 张图像、100 步 Token 轨迹、逐步日志及冻结几何资产。缺文件或一致性核验失败就停止，不会补跑。

不加载 VQ/扩散模型、不训练、不重新生成图像。仅用保存的 Token 做数值复核、读取既有图像制作审查页。原 G1 文件只读；派生表、图和 zip 写到 **Drive 独立的 `gds-v21/reviews/`** 目录，断开后仍保留。

输出覆盖：①落盘一致性；②匿名画质审查页；③t10/t0 保留率，区分重复事件与唯一位置；④70 个冻结几何竞争对的失败与晚期变化。画质需要人工看图评分；不会自动宣布 G2 通过。

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import base64, hashlib, io, json, os, subprocess, sys, uuid, zipfile
from google.colab import drive
drive.mount('/content/drive')
base = Path('/content/drive/MyDrive/CEG-WM/gds-v21')
source_dir = base / 'runs/g1-v21-20261128-8sources-48attempts'
assert (source_dir / 'report.json').is_file(), f'找不到 Drive 落盘报告：{source_dir}'
review_root = base / 'reviews'
review_root.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:6]
output_dir = review_root / ('g1-readonly-' + stamp)
log_path = review_root / ('g1-readonly-' + stamp + '.log')
payload = base64.b64decode('UEsDBBQAAAAIAAAAfF3aZsUU7wYAAKwQAAALAAAAZ2VvbWV0cnkucHmFV21v20YS/q5fMWfgTssexZCKYyRqFbSH5HpF2nxIAgMHgjBW5EokTHGJXTIVi/vxnZldSqTs9Azb0g7nfZ6ZWd7c3Pzb6D9UAz+/+7z6uo4SqAol65Udjjtdw0Hpo+rMEMFH3YFs4NPP/wI7NHlpdFP9oUx0c3Oz2Bt9hH3f5J3WtYXq2GrTQW36h1zmpVp4QiltWVe78dj0x3YAaaFpF4uPIXwI4befPn+ALSTx+jaE2/jNnfu/uI+R2rTRvq9rkSA1gRdAn0U3tGpLT2otu7vbYPHLu/cfv/zy5b8oIOIIOfEv5r9g8YmII0MI4k0U4uOYfgM8rq7OMR+T1+68APrxxNVIHQkxM87OKyZMxdavrpQjIVkT5zoKgsW7979++ekz+TiaXs+NrK4JMfOMRqYOxl4rst5dRTk7X0lGb648TPw5WCwWhdqDLAqjrFVWBBuWNfr3EHKECheoaooqx4fi5TqEl2sUYx7V9aYBcQvfgUABePECXgfwT5bj75GRX1UtRisHo/v2bEKbQhnUbxE1qhBGNgclPqCLj2rY1vK4KyTkGxDnWDzQIlvK9as7sVs6cK9Y7WZJhqNOP+yGDl1FpC131WEZBFFRHZTtBGrOvecHF5Y6tt0gPkzwVjWMNuZJ2cHMscrRPfg7InQa/sEHZ3Qnu0o3olAHo5T1UaJYrZwOfLA2sjgzTLWwCSMHkab4NddWsCD6vMKzrRp/zqalnfykM64QZkqybKxAh2HYvTZH0WqM1YZQej+7UnUyBJvLWoXQnfBvQLfLWaXJS+v8dPIBrCBZY0PAj5cEsKog+oKwYHVYGOLBj9Tpzbw3ucaRYZUwWAFVo4x3hZysWdeYfLJo0vUmI3iZNMlQ99meSeOMHDhzsjLmnnlPfCjunsZZOCpyhAQJ301MI5TqimAz5q7Ak2xydnfMWq5No4ydupmmcYjK8T+CFD/xP5/p01EwGdjfr7O5ezzqKMN11cj6EDVUpUu9vCV0mnL+DL3EqstTZbdJEB3l6eJ31Tw48I5Oe4M4SOAHrDBlBT/xJJsCojeOmDgizjImk0a5s6JM1xkZ4q8vsyAgrvUtmvrxvBgEMlvcIlsMN2AXrO5Nrh7GMTI6gksmpUWAHkmaBbmqa4wkuQPb79rqpGqbfY9+wlFhuKD7zuIa89qg0EdZNbypOCosS55fzas7HAPY0C7RdkB8n644aCP55/Qoz8eZlW5C3I6NIoTcImzs6fyEyCFsCEzRKxalVsFosTLCmG9qGJ7X4Ky7hnLeIdDyRyGwWYaxqqsk+P99qk11IPTQsvH6XH/ix9h7z7Qqbt1pozpVyK1J86gzjSLat9glc0qSMf9XWVcF2dUneLuFOIB/8PcfKC/uMMDbkT4wbAgeJFsVlPqXa8yT2/naCOHSqQfyDjeJtDSjBcZEI+DMhhauH1+N1d9LZZRg90KyhBM1GVvDqAZxJ/aVqotLUyOifq0aJQ1glVal7l5Yve+AubzI92cw+jSXWFjom4p60u+5MzKdoBsRfnp6i0/vOZyOvROhRdcq+Bsm9SN1hXePw5OVVXAv6169NwbzsHRGjr3tYKcg5btWcpctxwxb3rSzNuz6tqap8HzGBMlwJV2dGcmh84y2DTZ5deyPxIawwMUE9zFOHuxU4cfQuHNwzeJIfTJ/fPJpDw8Kr39BOrmFZFQOrP997LXYvqXr5bUSNzXFtyJzATivzgPR4sxUQu+sMl+5DS6lv57FFgOcMCJAJ9EQMml8jXrLodXYULayD62szPmiw4cQ1Cmv+0IRElIML3V9g4ABgzMaPl2qS7SSaOl0QwZML4jurpQ0PdLx3pttZheDEq24LLhYTk76RNJlMGPF4Wt504ol74klAbDB94LJ4gAcxQqW1BQPrZKPzDNZiCXvgdfP3028sK8gi16qiQMiuosdC0Fs7lu19+5tnqge0xnJtkUkCax8J8yWlvO2DL3Y1n0Ec62qRr3OaxenK9FTG0weDbDAuUNJ8ErqaVPyBRP+swHb6RZLqFnMVrVqunqAvKT0nt/GlrNGvEKNx9iur+riIddqv6/yCtX8Bcqe4tHfWVDYjSN808NLokB3BHPgsnGT5lujiTD0GPrcYd4Uvu0pQ+3g5C/JeHbimEsdioqdOLeTu9ZcmmvWDHgdIGsOjbNBSD+PSrWoCi3ixkXWjLt+xkKjt0CodIKjTx+zkPlJlEVCdojPGbY1GqGlTc09KwpLh9+oTacfVfPAA1Lw9zEf7jDfAJ7h/IbkYeXIs8kf4MJlyWYQXgybBhfp/2A8vuVX7OAvF0TVuBXNIm6Me8BN9lOH0BT3cUh2Ex/6uNm9LXSKjF0kU2bw70njKB9f9VIn5XmybDZlSXrxJ1BLAwQUAAAACAAAAHxdv63/ubIYAACkRwAADAAAAHJldmlld19nMS5wea087XLbxnb//RRwOlMA9goidW3XkYJMfeOPSXOtqk7unekwLAYkliQiEECwgCTadacP0Zk+UN+kT9LzsbtYgKSke6eeRCIXu2fP156vPdA333zzw9WfT6qy2HmNTDPz6SaXt1618lJvWW3rQrYy8942+Y30Pkwj71LeyMbLt3XVtMrbVpksYF4mo2+++eYJD3tps67TRknzfZOqTZEvzNffVFU+WTXV1qvTFh9ocN4VfDWT1KZr88J8a+Vde9uktfledtt656XKK+1QnZYZDMB/dcbQr378k4H84zZdW3S2aVsXVYsY9R+jTsnAf7Ne++H+vKje4ScCXrQMfS2rrWybndniJ/Hxzc8/iUuxbqquViLNskYqJZVYdHmRJctKrlb5Mpdlq0RTtWmbV6VQXY2rnzx58+njz/HMv/SF/8cJ/pjijzP4cYX/n7zz50+ePMnkyls16RLXBqlYhOdPPPjXyLZrSm9VVGkbpKeL0MtX3sKThZJ60C/TEihjCDAdsEAQCtAADFUrAWNC3OCtIYNUP0nUg66VsK6WMD87AR0oWxB7B7R4by7fgq50Zf57J0/qSuUI2SsQcHuiuuUSoMHcDZBJSoJgEYtN3qrgs94H/92kRZ7Fn5/GyMgL2E11RRuXdfRZNpUKLsWiqorQTufnM1o1jwn52Wf9dR7Hmg494KwiVvFiGt1MJ2IziQkd5sdsOpmHwh2A7zSXqYq/fL0gFqh4xpBXVeM11a2Xlx7xsieqjWF85rf5VuITf37hLTeVkiUSljZNugtgQrSWICN5J5fA5yxRspBL+OCLGSCStbtaxnnZ9rSDeAtZBgzJYaHzTAFEfh61VZGrNgjD8GnsLPMAaT1jm5dB+N3EHUnvgvD7+BKHSlD96fS7uP0ufjUZbkYsTXPQs7+kRSffNU3VBP6PJXHdU+kNGA9DjWeUQ/nhAIhWklVXxLz7zOV9Ow9nPDyfD5Yh03NiuV1/riU0A24FeThHrvVwZvl8uDELMUrrWpZZkOXLNjCCilth8HZ5RlqR3G5kmagUrSM/7TEIxR5/tCYmtDRtk3Y6IbxA8Wb9wnmkui3ISOxN17MPTt7bTN6BdUja6howBF1P8xL0yezosMKBFce92rvj4V+5yaP2OLxFyHLJM9UfCwUGQ2YBC3TvFFRF1k+dGanPe6UAWPPxohbMbhHjlnLm70nS59USV7Nm8CqYv02bXUz6YZQi0QZAA7Mndh+G6Ok1iwiPnq00ivJOGqAjtgaepz28QyjY+iZ6p8QeNFJO4EQ43ozMZWIdAWmI3VfveEBvD5FHWD68wSPgPwq8JvVeAvBkofxZu4RhwiOguEAeCWN0GA7h4/gVghnHoL7h3wr5AOBHwiVeJgO+t9MpmxfX/U37s39wibVfexbi0HRjv8zknhQwvEW+TBeFjNG3Bxbl0A1rWBOEPoY2hoEzD0FpYuKwoBEb7QnbJi1VQQFWbCKtoEEnHv3yj4G1GpvZ2fk8POn9MH0PT5vZlB1NuwHbFm9g4QmuvlDLFDDdwON+znKTlmvwArMy3UrSXvwgdlKhEs8C3yDgi3ShAgIZfj+VJ9+GIvAJIj+hjydT82zP5ga+Q5YvAOsiL9NiHZVVsw2cZxrCHAMBQGPuspJMmGUdGUf/uR/9VuXo4YgSCgv8PEMRtTtf2NnwFKgnAvpBZgn93Ee53+gu5jDUQRPFIRxUDswANTQh67JIlcpXu2SV5kXXyCAFxRXwYwI/JlruQDB8+X5ybsj12RAC/BXyyu9nTacHp4EVmk6TsiqPLxzAd2fy2gaiZciQwEHg90lSVKpNFjsXjlmcFph27ZJ9IJDKwFpf045RVLJUN0GWAusxZ9L0ghnYyhhHEbdc5SUc4HIp9cQsegu/3+OkkHMBd4gmhT0ciBJpE4Qv8jKTd/H7FBYNjiLNNCewA0tedc1SiqprIUHQWPFYjMmcfh5GYA2q4kYG4QVP5ad6Wf/UsFmDiHkCKiSP4JHisQgSTLQK+EzPwkCQZplnfaS6H6G+AWnsVK7M4m2nWm8hIVaFxWAEPcrxQNlhg3yNsoPsF44PiLetmp0OYDUu22t4EOhd41+aTgp5BxF3Ul27PFxIMA8SkgebAqH8E2S4PZNOLI/jsebgqX0+YtaAYRSoAxtwpcsgHKaxXIEeFrB2GMQze97Dk8uqfQ9ZXcZcWvkfczh15RqB5DqmBzbkJVB97n0xSH114vlruYNQryE1ivrjXRlVGKANkw3OzJxz/XsGT+axrhtEapOevXxlQALLFjtwU+BKoo28y/K1xORmnOHhbMtoMC2Zw2MzBysRET5TwQFR8F5YegisQ8JkPSZoPn+JEIZvNVcPUjoHh3Etfci5/A9TH1k4eth2ip6aMot/n8K+u6s5iXLLMh+mF15WEQsVSK9si52XNVXtoZWE5xBHtCbZSttWbmvQT8YCwi894M8N8uh+zSgg9uJ1j/TMX8tSNim7bT0nASIaCknN5LREN2qIm7vEcY6MorZb3Efvi9enALInFchZS8Tm9w4OYaaJIgLjL7Qn6VeSAzYCvqbN1p+H+5t+dYml5ZbSLylNT0SKC+jZ16cx5tFYnCG/iIu+KJqmRNJPAxiv76Pmz6U08mvAIchG47+H3rl7PIAMBI98HBL43E/854ZI8tenPkKxC2js11/3BxsryKpJYPfbtMlIUNPJ5JBROJTVIzSEdfn+HdeANDGU7ScmWwJfQ+cHfUp/uHz9OIJhPwxdYQyWW6Eg+z/n9fBpZHkhhuPAECxykFDuk8fPXJjgVVok3jZX27RdbvrTjAdrvDOpduwodhilRRGExtCO5u9zO0Ze60WPx5H3PUWej/A0AXCC5nJsM8nPJmT1kySMbvN2k2CAGvhmWVSDN3vArAIrBrs8tUYEAYP+zAbg5veR9cHUTbXPyvLVSjaqd7jG0IDbgRBDE0nVU4h98gbOXbxfUdWIQpAowSCwgbYo8+jIUvPgzCeYeAJmM6qSNZA54e9NyMkopBXsU2EaHSGz0Nk+YW4jlBH/aRKEVodYex+X3jfVZ1n2RWbadCz5oqquuxrLIMuqXELUUsL/wYx5ZeumlFUR+qG4FNMwnIsUopP4TKsPpsBcPgUEL3TqFtvyNYxR8M25Mm7WJDMuFU9fsetA4Eleriq3HJoLysgolS+7LYpUajTcGEevHNTgeDSLc9HorAPTL/jC2QbmXvAFkgrI2OjTDj79YT5MQDZ6KaZvYpP0uRt8gaUbXLrBpRtc2iQcyyW6IB/r36gOm2PPNqM07dmzw1lp2NtHHWJrqnXgfNqrKj2CiCUvuYDDdtIpQ+USi85cfk6a6pa+0SrzBTKo3yhK1VNvq0b1U1eQ2RWQ2LmiAvEsZCO0DxpK7ICzRusa63HjWy7qBpC+0zUyY537Wa53BiPdPyAXJggk/ujDOORHvuSjjE+e+6c8NDjIxBisuMczJ5hT7PgpBe99j4ZC0wlG4Q9ivEjVwBnIqlHpKaoGa9+A7wnnrleeNW45v3fhBBcsAFsRzKWDb78VJ1P4LzwYcDvnfeV/Qey+nhMUE+zkZe9k7Ho04R5m/xzSjoirm/wGxMZFIxWV9Wc/FOBqqtukzpfXcAQ4IcGrrNUQKaIpXvXEgVW8YDg0Sp/8ASvQ35nySQKRWVpw/VzQ4IgFaDs1WmAca/k0DsARikv3AdVro2tIP01e4Ocdxc4IEYLLgCd+Nwn/XX/8Hm+L/goGmzyGVntps9zAeR1oE9bA4i9Ys4KU9jMJuBWfERkMRJhA3jv82i9bQtrSL2PTPDt3OHEZCseQzj7PubQ4Dd0dePcoh9MBStiD1zUwyFviR93euVLitU9jPj5wFrvmBrSkAD99ryZS/GF3g0RDKpIKuSAtsdOiWrvXOVTmiJkZWDIj40P1P2LEGi+aaM4wEdSKlKtlUSnJM/hSShisaQwMiISf8CQBWwh5OZqTtiriM3kyPRMNfpw8XhcIpseQxs5VYwb0o+4fqGqitx8i507Q4fY9q08v71mfmBKvG0w9ihpYfIiUNl3Eg8h0diykjg9a7fDvg71QOx6Z8XAe5UW1BLn3+gBMwCwf+BvcR6U4Jt/9y82RrgBhVC0wikKfH9KJBzj5w89/Mc7G8vLc+wKg3VLH8UtfYvj44vcBOvSZaeda72k1cKf727T+XgoBuS9IJ4O/5wDgP76Wt7d6fD0/DLVAkpQI8gUg27f+GvDIlby5YhjzxUJ7Grs82D9ctNl/EDADSy+BsGpdQQIB7FLXDGx8jh7FH3LGus0hy1W6bqRkB6y94R7y91yT1wmE8FROIBzBB2YQk2HudIU/+vu98KKWx+diJ8hg9t5GOp+m/f6NQRnGXCU3Krl6l6jdlvU74fRLlkvJTKV7mPuWmgUHOfoornJ0cwLbrzHKWRXpesxe2zKAqZa/T+L/QwuFw62hFYPc/tI/TNqo7eJA3nZJ4T+EFVsIk03Nh9qq4HhnQBDgIJsbdqmHKIPgfLCJ5j7lA3AKtC47EZE5Z8+nc9MmwUc0PDDJTol5yt8owYPUeDd5RffwHoLWIsTxpivkiFI3Rxmkfs+ecSIhnD6M3u7pO6KRqTxwWbbsGlTRhHMfTI+4E8Qu1KEIrGW/0088stWMZswfaO+4x2JB7LZNQYzaKt1jqA71W7AiJeBdtSOoVslWpiXtaFRfT/onPwyHXsq5UD/3+tTxMOefPZOj5cgWUwOhvHXkk2N8wJUUihUuUrEQyzgYszDXBVkK7IPpVEwnYjJyADaVPYYcThAsI7q8Tvsvk3hhv0zi5T4fWdbOfVu8/C6eiFKmTUIZb47sbXMZ443sMvwunpKH5atRvJlrqwQgn6QHYPPFZEJV/3j/vhJZ4sqlT8vvo5TYrXXPpXrA2un0iHo6nBkumPQLeobxFOLRYXAPkLiHjdjbTgygO9ywxY1jzIBIWhyPHx8VSI94Y63KQYYa0zKYts/GQ9MGnDxootwbXyqJ2eYpWgM6afsvVLqVQIfCW8fe9yZc87AZnTvpauClE53JDxFQt6mzvHeaMGyt0G1aFOBGl1WZqT7Qd0fHUGEQk8UEG4mT68LCx8lUziIHxnOuC8wk/Hk4qmxQp25UgQ7sFzdyelaXazBwWMDItyNDtOWqyF5UBg9U/hkrDmcvXwn4n3wjjGL/8tPY//Thj/69uXDvyz1CwkNwp7h6kPtyaW2gwrayxSUDsOBg7Dm1QF4oEBqwgy+NCXSSSeyn5ntkhzt1gwqy8n/YyOU14PGFq3bPp19PX7yG5IQw9cWq6NSG1zpVw11si4/2pNnio87yBjczw4Jl7zLsor6JRzdR0rILu8T18HbRwO33+5Ch42aGYY10sCMqnuOgXQBsS0cQegNrQThevpZgW0mfDtDrwJs5nznnxXDdQrxyIgcXI+1N4tmSPN6S21R1pk2RUUIxqQrycll0mcRYliUaRsuq6Lal7tyBPe2dmr1UwgxrsQsoYA1nejMwRhARBFjTpP4NvrYcAzFZPZzeI/DorAQHFlqbhdbuISgzf2AT/Tkei2IXFOl2kaXe3TnZu7vvranb2xBbHWSbMC2h6U+x6uBMRL7v96ysGvl73OuWxY/jfGHjFeEPOqIwuVaJCy1K1+vAHkTXeGsmxKa24TwDsGjgsICV2vg18HuGCB+fmEtQRDbCltfkBg2Qslge2I4wXIKJgcgxnpHpIETne0wyHltzQf7eQaZ3jF9W/Y0hs1qOYf66gZFkidbHVfS/8/5YYLlWbSSkal4HFhQv8aj/Ba/wwHtj0wJQsdvmn6l/IPIuK/3mCDtSNMZYPIYNvBvZoN1UEUHHbpIjF8l6QQJTkqtPP/7lzS/vDlwqw9PBVTJ8j+iegS/2+7vx/e4Ej0tX0QLpAz2B40ltmQEBvO/yzlyUG5oG1CNMzTvDpGMk8s4aymlXEreyhF/ROUCsgQcIfvsK0TcDs5lvqCBtx/Wy8efzCLtFkqyjNky6oeTa/GwyNzCQTQaO5YVmn8uee68z/0VzAri8xToTNRWR+VWnBh2spakRZ8b6bITOa8CB1seUuW4qkGL8xUr4HDMUGjwk7a/2MqxG9473RlS0f+2QtcrXIr2D014XLdisBb4SpIIz8QcITNdoM+Ng+gfIaUbZU3onggTLeaG5QkAgUQOnrQhCgTykqilu/OzVeYC/n0/DZ6/meA3QIFeC8f2GaUqLnbYlyAP346TBKhJVfFTB/OcIw8jzuX8AAiia2XrQMvA0pmWDsUPlkTd0EqpO6TBqcPnP1m204zgopH1M9Ae8jfKt2lS3dL2guBiUb0c5JcxCR9LmbSGDIY2/lv5z83JZtMqLImAdmRl2zsWLs1CsIGolCX8bXgA0vD8P/Gq1GkR/a1CKmjfxP0yZ29YG6Pfq/vc//wsciJd2bbUFc7C0z+ls+/1G0xdD0G2+3rRJke7gKAThBe0GGgS/zeFY2dPB5H1hRTqfnH0lSYqszuPp2SS8QP01Fy1rY8bfSrVs8prKV+guZHaiOzR0eRjV/cIr49emc4O6bATVsf9hwg0SaDBeTCZcD2cTzqdmeGbseflWvDT+D01NsxTczdoHSzao6O8ihq+MKWqAjfpIQQcxeKQwABLjOOSJoxaIDoVJAiE9m04mE0yIr2UT+5Uv0qLepHH08qUo0oUsYkBQRymoAxu8Kg4mAsK1CuYvinR57YviNo5ek5YgSjtaGPjv6W6G6viYeebbbmv4+j//7eG+voXc66r/DqWu+X1SQKRdYM/pstqC0wyQ8ZC5eBmE4rrShjY87CEVco2JiFWq16Jc4rUB68/DKmWilcUuMa21VpFeHlakv0LcWCQF+pH9s+m5U01GgZqocja8b2q2+/KbYRlpLmakB8dfD9HRsbh3mpk1d9WAhY+9ZYaxeQlxSpvckSGwwr6DA32tND6OCuRb0JJpNHk5EPGdVo1/XiisoWYeiKquINTy0hW2o9HlA/aT2deH+rcr+XVKE1zR9HY6DYc6ZHTvI1DkQb5z0p9boN07BJf40d/57mnSoxXHgnhYY+pkI9PWCdRnTsxuM66ozm9A1uznHWMgdLIU9+E8B9CxE2qHdLIf1s/pC/HCNOtQvxC+BcnFHPQvCxUwshFvEfJbkqGgKiGvI1p6tzSYD1E7FhFiH80/YL5N69j/lP1r8aEErMEsxCf8JsUNwOWXKoYCZf3iyISveBA6IKC3Ie706mhsz6iXjMq5P771gsnJIlUyCw+anivwrkWBln3l2i7OvfHt3MyLqUOlwZ740zaHAWxXkS2pEvsFNE86ByHj5FsjEZHVXKRNgJijSCBO4oPmfxwZxgNKZ8GMFe+Kcx0ECux9WPuOvL/DPXp9Y7rpyKALymGLn2nRx3X3NFI+jfmT7bbWO5sw6VNX4p2HTSGAGmXmeFnXYMSc6rcWzj3/OTbZm1d2mJSgT9eoW43Q081S0S2kb5IbnahRKuu2tQqYOIqeyxb8gmlzx1jF7eWiln7dwS9sU3bC7dcvXguwhvkKNCChrfm1Q4YdChMwUhOqgjWG5ZS49kkDh/QQB+FLSe9M/pg2bb5Kl/giQ1nsLjB44tQRK30wQXh9Y7rAeyW6YqHXQvS+J/yuNy4DHJf6/Sm776bbQn4+TCzAmHB2fU41Tdje5LcpproYqnEqA8HiDRyMjLUdzKwO7EBkN7nCu/Xfumy9xdvSfscPZ1jjyxXWrc1GCb6C1Zq8plo5CNmX5SCm475+rjY5EAdVqPjB6hVmVSRcLMSB3H2+z4Pkqwd5mzbISBXP/E/6tX/XY5A66SslkJH0yA7g9V92sfe3APoeohwmF6rSLxXILPL3L1I8z3/jmbeKtUfapju6Ia4Bk4aEAOeMvlz0bnOwDcnDW6adAiGYjidv3WEzVivlkY2v0DRqgzWmi1IRTBnNa0beW371ggrmfQiMqQmcJxQs/pUEfrlYHdnwo44Fq5XnVMk9DEKMrR0mSVyH5koC/ZEO/fKyiSqP7PMDl935b3z89CcAQ1VHhen9SX9VCzxum3zRcZBB5/QIwEs3iflwBrQr5bUbYNumKjK0kWA5scEuy1DM85GJ0jWMe40TzxkbJwvi07s3bz++i9q7dgjB/5nDHPs3TUBQEv9+SWaSsKFNiSAHtBSCsqeZN6otCm+/RC68vdIv6eXxittwH1MR0XaFzAwql02QbQWNyyQX2uwcqYC4KjcmSatpXoIWkUdGzqSLnEp8XlMtOtWWcK6FTUbxJRs4LtgVQh3vrqDh2IHRg/z/Vxsecg9lzH+yBWKia5nosQD9hn7N77mvkfaF/zmvffPWoC7XYDkYuE8Csu/ZndtZF8MZGj48158GrynyPk+ePAE3m9D7DkkCpi9J8AY+SXR3Rx2bP1QTvWnWHVrpK/zWQEwL0V6WARU8HPgnOnjG92H5DSS+YzkwkREeT9ScAoNaR7QnrsHcAV+jxI+6AU/QZ8Oa/wNQSwECFAMUAAAACAAAAHxd2mbFFO8GAACsEAAACwAAAAAAAAAAAAAAgAEAAAAAZ2VvbWV0cnkucHlQSwECFAMUAAAACAAAAHxdv63/ubIYAACkRwAADAAAAAAAAAAAAAAAgAEYBwAAcmV2aWV3X2cxLnB5UEsFBgAAAAACAAIAcwAAAPQfAAAAAA==')
assert hashlib.sha256(payload).hexdigest() == '9b640bfa7e9acace73e0e61d55e4b2cf0cc16ddf9b349e2c664d1ad813b319bb'
task_dir = Path('/content/g1-review-tools-9b640bfa7e9a')
task_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for member in archive.infolist():
        assert Path(member.filename).name == member.filename
        (task_dir / member.filename).write_bytes(archive.read(member))
print('只读输入：', source_dir)
print('独立持久化输出：', output_dir)
print('日志：', log_path)


### 从 Drive 重读并核验

逐一检查 48 份结果：文件完整、100 步齐全、Token 与日志/指标/CSV 一致，重新计算的几何资产与原 hash 一致。核验后生成 8 页匿名画质图、保留率表和失败竞争对表；所有已读取的原始文件在分析前后比较 SHA256。

通常只需 CPU 和 Colab 预装的 NumPy/Pandas/Matplotlib/Pillow；本单元不安装生成模型库。若失败，回传下面打印的 traceback 和 Drive 日志。

In [ ]:
env = os.environ.copy()
env['MPLBACKEND'] = 'Agg'
env['PYTHONUNBUFFERED'] = '1'
command = [sys.executable, str(task_dir / 'review_g1.py'),
           '--source', str(source_dir), '--output', str(output_dir)]
with log_path.open('w') as log:
    proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env=env)
    try:
        for line in proc.stdout:
            print(line, end='', flush=True); log.write(line); log.flush()
        returncode = proc.wait()
    except BaseException:
        proc.terminate()
        try: proc.wait(timeout=10)
        except subprocess.TimeoutExpired: proc.kill(); proc.wait()
        raise
if returncode:
    raise RuntimeError(f'分析未完成。原始结果不补跑；检查日志：{log_path}')
review = json.loads((output_dir / 'review.json').read_text())
print(json.dumps({k: review[k] for k in ['completed_units', 'verified_inputs',
    'original_files_unchanged', 'human_quality_review', 'G2_decision']}, ensure_ascii=False, indent=2))


### 画质：先看匿名页

每页 6 张既有图像，显示匿名 ID 和提示词，不显示方法。请检查明显结构缺陷、过强纹理/色偏、提示符合度；不要用 Token 命中率代替画质。

`quality_review_copy.csv` 是独立评价表副本，不覆盖原表。两位评价者应分别填写 1–5 分画质、1–5 分提示符合度、0/1 严重缺陷；有条件时独立评价，不先共享判断。仅一人检查或已看过方法标签时应说明，不能声称完成严格双人盲评。

In [ ]:
from IPython.display import display, Image as DisplayImage
for page in sorted(output_dir.glob('quality_blind_*.png')):
    print(page.name)
    display(DisplayImage(filename=str(page)))
print('评价表副本：', output_dir / 'quality_review_copy.csv')


### t10/t0：参考是否保留下来

`unique_group_retention` 以每个位置最后一次成功写入的组符号为参考，每位置只计一次；`event_group_retention` 按成功事件计数，同一位置可能重复。组仍正确但 Token 已变，不等于参考丢失；端点正确也不能证明中间从未消失。N 没有写入事件，其保留率为空值。

下面的平均数以源为单位；不要把逐步事件数当作独立样本量。

In [ ]:
import pandas as pd
arms = pd.read_csv(output_dir / 'arm_summary.csv')
cols = ['arm', 'sources', 'final_address_hit_fraction', 'min_margin_t0',
        'unique_success_positions', 'unique_group_retention_t10', 'unique_group_retention_t0',
        'unique_exact_token_retention_t0', 'event_group_retention_t0']
display(arms[cols])
display(DisplayImage(filename=str(output_dir / 'retention.png')))
display(DisplayImage(filename=str(output_dir / 'margin_by_source.png')))


### 失败的几何竞争对

P 每个源最终最差的**同一个**竞争对，会分别列出 t11、t10、t0 间隔。正值代表该固定比较中真几何得分更高；≤0 表示错误几何胜出或打平。这是理想 Token 符号链，不是 RGB 同步成功率。

`positive_at_11_nonpositive_final` 表示写入结束时该对为正、最终变为非正；`already_nonpositive_at_11_and_10` 表示在这两个端点就未解决。中间恢复后再失败另列。最终最差对属于事后诊断，不能当作事先独立测试集。

参数顺序是 `(角度°, 尺度, tx像素, ty像素)`，数学 y 轴向上。输出有效支持比例不等同裁剪后保留的源图面积。

In [ ]:
worst = pd.read_csv(output_dir / 'P_worst_pairs.csv')
for prefix in ['r', 'h']:
    worst[prefix + '_transform'] = worst.apply(lambda row: tuple(round(float(row[prefix + '_' + p]), 4)
        for p in ['angle', 'scale', 'tx', 'ty']), axis=1)
display(worst[['source_id', 'pair_id', 'r_transform', 'h_transform', 'relative_type',
               'margin_t11', 'margin_t10', 'margin_t0', 'failure_stage']])
frequency = pd.read_csv(output_dir / 'failure_pair_frequency.csv')
display(frequency[(frequency.arm == 'P') & (frequency.nonpositive_sources > 0)].head(15))
display(DisplayImage(filename=str(output_dir / 'P_pair_heatmap.png')))


### 持久化结果与回传

原始 G1 目录不会改变。所有派生结果及 zip 已放在 Drive 的独立审查目录。请回传 `*_review.zip`（包含图页、表和核验摘要），或先回传 `arm_summary.csv`、`P_worst_pairs.csv` 和画质观察。包内匿名图像页是原有 48 张生成图的汇集，请按需要分享。

**到此停止，不启动 G2。** 要结合画质、端点保留和失败类型再判断；本 notebook 没有自动设置新的“过关阈值”。

In [ ]:
archive_path = Path(str(output_dir) + '_review.zip')
print('审查目录（Drive 持久保存）：', output_dir)
print('回传压缩包（Drive 持久保存）：', archive_path)
print('原始 G1 输入未变：', review['original_files_unchanged'])
print('G2：尚未启动，等待收尾审查判断。')
# 如需下载到本机，可取消下一行注释；不会修改原始 G1 文件。
# from google.colab import files; files.download(str(archive_path))
